# Watch-history exploration

Descriptive analysis of viewing behavior and drama metadata. This notebook examines rating distributions, genre and cast associations, and completion patterns to inform subsequent recommendation experiments.

**Scope:** one recorded viewing history. Findings describe that history; they do not establish causal effects or predict enjoyment.

| Input | Output |
|---|---|
| Enriched watch-history CSV following the project schema | Summary tables, distributions and association plots |
| JSON-array genre and main-cast fields | Separate drama–genre and drama–actor summaries |

## Data and execution

Install the package with notebook dependencies using `python -m pip install -e ".[notebook]"` from the repository root. Supply `data/processed/watch_history_enriched.csv` through the documented import and enrichment workflow. Personal datasets are not distributed with this repository.

Copy this notebook to `notebooks/local/` before executing against personal data. The published copy intentionally contains no saved results. Execution uses local snapshots only and does not modify source data.

Genres and cast are expanded independently: each drama counts once within each membership, while totals across groups may exceed the number of dramas. Missing ratings are excluded from rating denominators. An 8+ threshold is used for descriptive comparisons; charts restrict selected genre rankings to groups with at least five records, without implying statistical significance.


In [ ]:
from pathlib import Path
import os
import hashlib
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.environ.setdefault("MPLCONFIGDIR", str(root / "data/processed/.matplotlib"))
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from dr_drama.analysis.eda import load_history, memberships, grouped_summary, correlations

source = root / "data/processed/watch_history_enriched.csv"
source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
if not source.exists():
    raise FileNotFoundError("Prepare the enriched watch-history CSV using the README import and enrichment instructions.")
history = load_history(source)
genre_rows = memberships(history, "genres")
actor_rows = memberships(history, "main_cast")
genres = grouped_summary(genre_rows, "genres")
actors = grouped_summary(actor_rows, "main_cast")
completed = history[history.completed]
MIN_GENRE_N = 5
pd.set_option("display.max_rows", 100)
plt.rcParams.update({"figure.figsize": (9, 5), "axes.spines.top": False, "axes.spines.right": False})

def bar(series, title, xlabel, color="#416d9c"):
    series = series.sort_values()
    fig, ax = plt.subplots(figsize=(9, max(4, len(series) * .32)))
    series.plot.barh(ax=ax, color=color)
    ax.set(title=title, xlabel=xlabel, ylabel="")
    fig.tight_layout()
    display(fig)
    plt.close(fig)

def scatter(x, title):
    fig, ax = plt.subplots()
    for status, group in history.groupby("status"):
        ax.scatter(group[x], group.my_rating, label=f"{status} (n={len(group)})", alpha=.65)
    ax.set(xlabel=x.replace("_", " "), ylabel="Personal rating", title=title, ylim=(.5, 10.5))
    ax.legend(fontsize=8)
    fig.tight_layout()
    display(fig)
    plt.close(fig)

def subset_genres(mask):
    ids = set(history.loc[mask, "external_api_id"])
    counts = genre_rows[genre_rows.external_api_id.isin(ids)].groupby("genres").external_api_id.nunique()
    return counts

print(f"Source: {source.name}; SHA-256: {source_hash}")
print(f"Records: {len(history)}; genre memberships: {len(genre_rows)}; actor memberships: {len(actor_rows)}")
if "mdl_match_status" in history:
    display(history.mdl_match_status.value_counts().rename("records").to_frame())


## Viewing coverage

Counts distinct dramas by recorded status, separating completed viewing from dropped, unfinished and currently watching records. These counts measure recorded exposure, not viewing time.

In [ ]:
status_counts = history.status.value_counts().reindex(["completed", "dropped", "unfinished", "watching"], fill_value=0)
display(status_counts.rename("dramas").to_frame())
display(Markdown(f"Recorded exposure: **{history.external_api_id.nunique()} dramas**; completed: **{len(completed)}**."))
bar(status_counts, "Recorded viewing status", "Unique dramas")

## Rating distribution

Summarizes personal ratings overall and by status. Mean, median and the histogram describe both central tendency and variation; partially watched dramas remain visible as a separate group.

In [ ]:
display(history.my_rating.describe().to_frame("personal_rating"))
display(history.groupby("status").my_rating.agg(["count", "mean", "median"]).round(2))
display(Markdown(f"Average **{history.my_rating.mean():.2f}/10**; median **{history.my_rating.median():.1f}/10**."))
fig, ax = plt.subplots()
ax.hist(history.my_rating.dropna(), bins=np.arange(.5,11,1), edgecolor="white", color="#416d9c")
ax.set(xlabel="Personal rating (one-point display bins)", ylabel="Drama count", title="Personal-rating distribution", xticks=range(1,11))
fig.tight_layout(); display(fig); plt.close(fig)

## Highly rated completed dramas

Reports the share of rated, completed dramas receiving at least 8/10. The denominator excludes missing ratings and incomplete records; this proportion is conditional on completion.

In [ ]:
rated_completed = completed[completed.my_rating.notna()]
high_completed = int(rated_completed.high_rating.sum())
pct_completed = high_completed / len(rated_completed) * 100 if len(rated_completed) else np.nan
display(Markdown(f"**{high_completed}/{len(rated_completed)} rated completed dramas ({pct_completed:.1f}%)** received 8+. Completed dramas missing ratings: {completed.my_rating.isna().sum()}."))

## Genre coverage

Ranks genres by distinct-drama count. Frequency measures exposure and should not be interpreted as preference strength.

In [ ]:
display(genres[["dramas"]].sort_values("dramas", ascending=False))
bar(genres.dramas.nlargest(12), "Most frequently watched genres", "Dramas with genre (groups overlap)")

## Ratings by genre

Compares group means and medians alongside sample sizes. The chart shows groups with sufficient display support; all groups remain in the table. Overlapping membership limits independent comparisons.

In [ ]:
genre_rank = genres[genres.dramas >= MIN_GENRE_N].sort_values("mean_rating", ascending=False)
display(genres[["dramas", "mean_rating", "median_rating", "small_sample"]].sort_values("mean_rating", ascending=False).round(2))
bar(genre_rank.mean_rating.head(10), "Average rating by genre (n ≥ 5)", "Mean personal rating")
display(Markdown(f"Among genres with n ≥ 5, **{genre_rank.index[0]}** ranks highest: {genre_rank.iloc[0].mean_rating:.2f}, n={int(genre_rank.iloc[0].dramas)}."))

## High-rating rates by genre

Calculates the proportion of rated dramas receiving 8+ within each genre. Numerators and denominators are reported explicitly; small-group percentages are unstable.

In [ ]:
display(genres[["dramas", "rated", "high_rated", "high_rating_pct", "small_sample"]].sort_values("high_rating_pct", ascending=False).round(1))
bar(genres[genres.dramas >= MIN_GENRE_N].high_rating_pct.nlargest(10), "Genre 8+ rates (n ≥ 5)", "Percent of rated dramas")

## Drop rates by genre

Compares explicit dropped status against all recorded dramas within each genre. Unfinished records are not relabeled as dropped. Sparse drop counts limit interpretation.

In [ ]:
display(genres[["dramas", "dropped", "drop_pct", "small_sample"]].sort_values(["drop_pct", "dramas"], ascending=False).round(1))
bar(genres[genres.dramas >= MIN_GENRE_N].drop_pct.nlargest(10), "Observed genre drop rates (n ≥ 5)", "Dropped / recorded dramas (%)", "#ad6553")

## Public and personal ratings

Uses scatter plots and Pearson/Spearman correlations on complete pairs, with a completed-only comparison. Correlation measures association, not causation. Current public scores are not historical snapshots from viewing time.

In [ ]:
public_corr = correlations(history, "public_rating")
display(pd.DataFrame({"all_records": public_corr, "completed_only": correlations(completed, "public_rating")}))
scatter("public_rating", "Public versus personal rating")
display(Markdown(f"All records: Pearson **{public_corr.pearson:.2f}**, Spearman **{public_corr.spearman:.2f}**, paired n={int(public_corr.paired_n)}."))

## Release-year patterns

Compares ratings across release years and reports overall associations. Release year is distinct from viewing year and cannot establish changes in taste over time.

In [ ]:
year_summary = grouped_summary(history, "release_year")
display(year_summary[["dramas", "mean_rating", "median_rating"]].round(2))
year_corr = correlations(history, "release_year")
display(year_corr.to_frame("value")); scatter("release_year", "Release year versus personal rating")

## Episode count and completion

Groups dramas into episode-count bands and compares ratings and completion. Counts may reflect broadcast segments or separate parts; they are not directly comparable measures of total runtime.

In [ ]:
length_history = history.copy()
length_history["length_group"] = pd.cut(length_history.number_of_episodes, bins=[0,12,16,np.inf], labels=["Short: ≤12", "Medium: 13–16", "Long: ≥17"])
length_summary = grouped_summary(length_history, "length_group")
display(length_summary[["dramas", "mean_rating", "completed", "completion_pct", "dropped", "drop_pct"]].round(2))
display(correlations(history,"number_of_episodes").to_frame("value"))
scatter("number_of_episodes", "Episode count versus personal rating")
bar(length_summary.completion_pct, "Completion by episode-count group", "Completed / recorded dramas (%)")

## Cast in highly rated dramas

Counts main-cast appearances in 8+ dramas while retaining total exposure. Frequent appearances are evidence of overlap, not an isolated actor effect.

In [ ]:
high_actor_rank = actors[actors.high_rated > 0].sort_values(["high_rated", "dramas"], ascending=False)
display(high_actor_rank[["dramas", "high_rated", "high_rating_pct", "mean_rating"]].head(20).round(2))
bar(high_actor_rank.high_rated.head(12), "Main-cast appearances in 8+ dramas", "Distinct highly rated dramas")

## Recurring cast

Identifies actors appearing in multiple recorded dramas. Counts are limited by provider coverage and name consistency; supporting credits may be absent.

In [ ]:
repeat_actors = actors[actors.dramas >= 2].sort_values(["dramas", "mean_rating"], ascending=False)
display(repeat_actors[["dramas", "mean_rating", "high_rated", "high_rating_pct"]].round(2))
display(Markdown(f"**{len(repeat_actors)} main-cast names** appear in at least two recorded dramas."))

## Favorite-drama characteristics

Compares favorites with other records and reports genre prevalence within each group. Favorite status is post-watch feedback and is not an independent validation target for personal ratings.

In [ ]:
favorite_summary = history.groupby("favorite").agg(dramas=("external_api_id","nunique"), mean_rating=("my_rating","mean"), mean_public_rating=("public_rating","mean"), median_episodes=("number_of_episodes","median"), median_year=("release_year","median"))
display(favorite_summary.round(2))
fav_mask = history.favorite.eq("yes")
fav_counts = subset_genres(fav_mask)
other_counts = subset_genres(~fav_mask)
favorite_genres = pd.DataFrame({"favorite_n":fav_counts, "other_n":other_counts}).fillna(0)
favorite_genres["favorite_pct"] = favorite_genres.favorite_n / fav_mask.sum() * 100
favorite_genres["other_pct"] = favorite_genres.other_n / (~fav_mask).sum() * 100
favorite_genres["difference_pp"] = favorite_genres.favorite_pct - favorite_genres.other_pct
display(favorite_genres.sort_values("favorite_n",ascending=False).round(1))
display(pd.crosstab(history.favorite, history.would_rewatch))
bar(favorite_genres.favorite_pct.nlargest(10), "Genre prevalence among favorites", "Percent of favorite dramas")

## Dropped-drama characteristics

Summarizes explicitly dropped titles and contrasts their genre composition with the remaining history. Genre prevalence among dropped shows differs from the within-genre drop rate above.

In [ ]:
dropped = history[history.dropped]
display(dropped[["title","my_rating","genres","number_of_episodes","public_rating"]])
display(history.groupby("status").agg(dramas=("external_api_id","nunique"), mean_rating=("my_rating","mean"), median_episodes=("number_of_episodes","median"), mean_public_rating=("public_rating","mean")).round(2))
drop_counts = subset_genres(history.dropped)
rest_counts = subset_genres(~history.dropped)
drop_genres = pd.DataFrame({"dropped_n":drop_counts,"other_n":rest_counts}).fillna(0)
drop_genres["share_of_dropped_pct"] = drop_genres.dropped_n / history.dropped.sum() * 100
drop_genres["share_of_other_pct"] = drop_genres.other_n / (~history.dropped).sum() * 100
display(drop_genres.sort_values("dropped_n",ascending=False).round(1))

## Implications and limitations

Genre and actor associations identify candidate preference signals for later ranking experiments. Public ratings provide a reference baseline. Year and episode count require independent evidence of usefulness before inclusion in a model.

Selection bias, overlapping attributes, sparse groups, provisional title matches and incomplete credits constrain interpretation. Favorite flags, rewatch responses and final status are post-watch outcomes and must not enter a target drama's pre-watch features. Historical affinity features must be constructed from training data only.

No model is fitted here. The next notebook compares public-quality ranking with content similarity using held-out feedback.


In [ ]:
assert hashlib.sha256(source.read_bytes()).hexdigest() == source_hash, "Source snapshot changed during analysis"